# Transform Results Data
1. Read bronze `results` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `results` table

In [0]:
%run ../00_common/01_configuration

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 & 4 - Read bronze `results` table, select only the required columns and standardise column names

In [0]:
results_df = (
  spark.table(bronze_table)
       .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingestion_timestamp",
                "source_file")
       .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

#### Step 5 & 6 Apply Data Quality Checks 
- Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
- Remove duplicate records

In [0]:
results_valid_df = (
    results_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull() 
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [0]:
display(results_df.count() - results_valid_df.count())

108

#### Step 7 - Transform values of column `race_name` to Title Case

In [0]:
results_final_df = (
    results_valid_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

#### Step 8 - Write the transformed data to silver `results` table

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

season,round,constructor_id,driver_id,race_date,race_name,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file
1950,1,alfa,fagioli,1950-05-13,British Grand Prix,2,70,3,6.0,2,2,Finished,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,alfa,fangio,1950-05-13,British Grand Prix,3,62,1,0.0,12,R,Oil leak,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,alfa,farina,1950-05-13,British Grand Prix,1,70,2,9.0,1,1,Finished,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,alfa,reg_parnell,1950-05-13,British Grand Prix,4,70,4,4.0,3,3,Finished,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,alta,crossley,1950-05-13,British Grand Prix,17,43,24,0.0,16,R,Transmission,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,era,gerard,1950-05-13,British Grand Prix,13,67,12,0.0,6,6,+3 Laps,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,era,peter_walker,1950-05-13,British Grand Prix,10,5,9,0.0,20,R,Gearbox,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,era,rolt,1950-05-13,British Grand Prix,10,5,9,0.0,20,R,Gearbox,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,lago,cabantous,1950-05-13,British Grand Prix,6,68,14,3.0,4,4,+2 Laps,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,1,lago,claes,1950-05-13,British Grand Prix,21,64,18,0.0,11,11,+6 Laps,2026-08-10T19:09:31.330991Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
